# ROPE-TIEGCM (f10, f10_41day_avg, ap): add-ons demo

The same `rope` interface as v0.6 (`forecast` → cache → `get` / `get_batch`), with two optional add-ons:

| | v0.6 behaviour (default) | add-on |
|---|---|---|
| initial condition | IC table (+ 48 h warm-up) | `use_da=True`: assimilated state at the start time |
| F10.7 after the start | driver data | `use_f10_forecaster=True`: F10.7 forecaster ensemble |
| uncertainty | model (ensemble) spread | + input uncertainty of whatever is on (unscented transform) |

Switch them per call (below) or in `config/rope.conf` (`[data_assimilation] enabled`, `[f10_forecaster] enabled`).
Data-assimilation states cover **2024-02-29 .. 2024-08-01** (hourly). Details: `docs/addons.md`.

Run from the package's `demo/` folder; it only uses `python/rope.py` (no extra packages except numpy/matplotlib).

In [ ]:
import os, sys, time, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PKG = Path.cwd().parent if Path.cwd().name == "demo" else Path.cwd()   # archive root
sys.path.insert(0, str(PKG / "python"))
from rope import Rope, ROPE_INTERP

CACHE = str(PKG / "demo" / "demo_forecast_grid.bin")      # keeps the demo's cache next to it
r = Rope(config_path=PKG / "config" / "rope.conf", cache_path=CACHE)
r.print_model()

## 1. v0.6 behaviour: add-ons off

In [ ]:
START, H = "2024-05-10 18:00:00", 72
t0 = time.time()
res_off = r.forecast(START, H, use_da=False, use_f10_forecaster=False)
print(f"{time.time() - t0:.1f} s ", res_off)
with r:
    print("one point:", r.get(time="2024-05-11T06:30:00", lst=14.0, lat=30.0, alt_km=450.0))

## 2. Data assimilation + F10.7 forecaster

`addons` in the result shows the assimilated state used and the daily F10.7 forecast. The uncertainty now
combines the model spread with the input (initial-condition + F10.7) uncertainty: 2(10+1)+1 = 23 input sigma points.

In [ ]:
t0 = time.time()
res_on = r.forecast(START, H, use_da=True, use_f10_forecaster=True)
print(f"{time.time() - t0:.1f} s")
a = res_on["addons"]
print("DA state:", a["data_assimilation"])
print("uncertainty:", a["uncertainty"])
print(f"F10.7 forecaster: {a['f10_forecaster']['ensemble']}, sigma = {a['f10_forecaster']['sigma_sfu']:.2f} sfu")
for d in a["f10_forecaster"]["daily"]:
    print(f"   {d['date']}  {d['mean']:7.2f} ± {d['std']:5.2f} sfu")

## 3. Compare: global-mean density and uncertainty at 400 km (batch queries)

In [ ]:
LST = np.arange(0.0, 24.0, 1.0); LAT = np.arange(-80.0, 81.0, 10.0)
def global_mean(start, horizon, alt=400.0, **switches):
    r.forecast(start, horizon, **switches)
    t_start = np.datetime64(start.replace(" ", "T"))
    hours = [str(t_start + np.timedelta64(h, "h")) for h in range(horizon + 1)]
    L, A = np.meshgrid(LST, LAT)
    n = L.size
    rho, sig = [], []
    with r:
        for t in hours:
            pts = r.get_batch(times=[t] * n, lsts=L.ravel().tolist(), lats=A.ravel().tolist(),
                              alts_km=[alt] * n, mode=ROPE_INTERP)
            w = np.cos(np.deg2rad(A.ravel()))
            rho.append(np.sum(w * [p["density"] for p in pts]) / w.sum())
            sig.append(np.sum(w * [p["uncertainty"] for p in pts]) / w.sum())
    return np.array(hours, dtype="datetime64[s]"), np.array(rho), np.array(sig)

runs = {
    "v0.6 (IC table, observed F10)": dict(use_da=False, use_f10_forecaster=False),
    "data assimilation":             dict(use_da=True,  use_f10_forecaster=False),
    "DA + F10 forecaster":           dict(use_da=True,  use_f10_forecaster=True),
}
curves = {k: global_mean(START, H, **v) for k, v in runs.items()}

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for (name, (t, m, s)), c in zip(curves.items(), ["k", "tab:blue", "tab:green"]):
    ax[0].plot(t, m, color=c, lw=2, label=name)
    ax[1].plot(t, s, color=c, lw=2, label=name)
ax[0].set_ylabel("global-mean ρ @ 400 km (kg/m³)"); ax[0].legend(); ax[0].set_title(f"start {START}")
ax[1].set_ylabel("global-mean σ @ 400 km (kg/m³)"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Same thing from the command line

```bash
bin/rope forecast --start "2024-05-10 18:00:00" --horizon 72 --da on --f10-forecaster on
bin/rope get --mode interp --time "2024-05-11T06:30:00" --lst 14 --lat 30 --alt 450
```

GPU (CUDA 12 archive): set `decoder.device = cuda` in `config/rope.conf`; choose the card with `CUDA_VISIBLE_DEVICES`.

In [ ]:
r.shutdown()
os.remove(CACHE)